# Fretboard assignment: CAGED-box + second-order position prior (combined v2)

This notebook combines two independent improvements explored separately:

1. **`combined_all_tuned_v2`** (second-order position prior): conditions the
   empirical `P(string, fret | midi)` prior on a coarse bucket of the
   *previous onset group's* average string register (low/mid/high).
2. **CAGED-box Viterbi** (Ani's approach): makes the Viterbi *state* a
   4-fret "hand position window" anchored to real CAGED pentatonic boxes
   for the detected key, instead of tracking only the immediately previous
   note.

`assign_combined_caged_v2` below uses CAGED's hand-position-window as the
state space (so moving the hand costs something, and notes far outside the
active window are penalized), but scores *which* position to use within
that window using the learned first- and second-order position priors from
v2.

Leave-one-out cross-validation (LOOCV) is run over the local GuitarSet
sample so all four methods (`combined_all_tuned`, `combined_all_tuned_v2`,
`caged_box`, `combined_caged_v2`) get directly comparable, leakage-free
`exact_position_acc` numbers.

In [ ]:
from pathlib import Path
from itertools import product
from collections import defaultdict
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Google Drive mount.')

## Config

In [ ]:
CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'fretboard_playability_combined_caged_v2'
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    OUTPUT_DIR = Path('fretwork_outputs_combined_caged_v2')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

MAX_FRET = 24
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4
STRING_NAMES = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
ONSET_TOLERANCE_SECONDS = 0.035

COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25

# --- CAGED box constants (from Ani's caged_box notebook) ---
BOX_WINDOW = 4          # a hand position spans this many frets (anchor .. anchor+window)
SHIFT_FREE = 2          # repositioning the hand by <= this many frets is free
BOX_CENTER_COST = 0.15  # mild pull toward the centre of the active window
BOX_OUTSIDE_COST = 3.00 # penalty per fret a note sits OUTSIDE the active window
OPEN_OUT_OF_BOX_COST = 0.60  # using an open string while parked in a high box breaks the shape
BOX_OFFBOX_COST = 0.60  # prefer window anchors that line up with a real pentatonic box
BOX_NONHOME_COST = 1.00 # prefer the "home" box (root on the low E string)
BOX_LOWNECK_COST = 0.04 # tiny tiebreak toward lower neck positions

CAGED_WEIGHTS = {
    'box_window': 1.00,  # how hard to respect the active CAGED window
    'hand_move': 0.70,   # cost of shifting the hand position between groups
}

print(f'OUTPUT_DIR: {OUTPUT_DIR.resolve()}')

## Fretboard layout, key/chord theory, JAMS parsing

(identical to `fret_algo_combined_tuned_v2_experiments.ipynb` -- kept here
so this notebook is self-contained.)

In [ ]:
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for string_idx, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({
                'string': string_idx,
                'string_name': STRING_NAMES[string_idx],
                'fret': fret,
                'midi': midi,
                'pitch_class': midi % 12,
            })
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']),
        'string_name': row['string_name'],
        'fret': int(row['fret']),
        'midi': int(row['midi']),
        'pitch_class': int(row['pitch_class']),
    })

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p['fret'] <= max_fret]

print('Fretboard rows:', len(fretboard_df))

In [ ]:
PITCH_CLASS_NAMES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2, 2, 1, 2, 2, 2, 1]
MINOR_STEPS = [2, 1, 2, 2, 1, 2, 2]
MAJOR_QUALITIES = ['maj', 'min', 'min', 'maj', 'maj', 'min', 'dim']
MINOR_QUALITIES = ['min', 'dim', 'maj', 'min', 'min', 'maj', 'maj']

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]
    cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12
        pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name:
            continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            qualities = MAJOR_QUALITIES if mode == 'major' else MINOR_QUALITIES
            chords = []
            for degree, (pc, qual) in enumerate(zip(scale_pcs, qualities), start=1):
                chords.append({
                    'degree': degree,
                    'root_pc': pc,
                    'root': PC_TO_NOTE[pc],
                    'quality': qual,
                    'symbol': f'{PC_TO_NOTE[pc]}:{qual}',
                })
            rows.append({
                'key': f'{root_name} {mode}',
                'root': root_name,
                'root_pc': root_pc,
                'mode': mode,
                'scale_pcs': scale_pcs,
                'scale_notes': [PC_TO_NOTE[pc] for pc in scale_pcs],
                'diatonic_chords': chords,
            })
    return pd.DataFrame(rows)

key_db = build_key_database()

In [ ]:
CHORD_INTERVALS = {
    'maj': [0, 4, 7], 'min': [0, 3, 7], 'dim': [0, 3, 6], 'aug': [0, 4, 8],
    '7': [0, 4, 7, 10], 'maj7': [0, 4, 7, 11], 'min7': [0, 3, 7, 10], 'm7': [0, 3, 7, 10],
    'sus4': [0, 5, 7], 'sus2': [0, 2, 7], '5': [0, 7],
}
QUALITY_ALIASES = {'M': 'maj', 'major': 'maj', '': 'maj', 'm': 'min', 'minor': 'min', 'dom7': '7'}

def normalize_quality(q):
    if q is None:
        return 'maj'
    q = str(q).strip()
    return QUALITY_ALIASES.get(q, q)

def chord_tones(root_pc, quality='maj'):
    quality = normalize_quality(quality)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return sorted({(root_pc + i) % 12 for i in intervals})

def parse_chord_symbol(symbol):
    if symbol is None:
        return None
    s = str(symbol).strip()
    s = s.split('/')[0]
    if s in ['N', 'X', 'nan', 'None', '']:
        return None
    if ':' in s:
        root, qual = s.split(':', 1)
    else:
        m = re.match(r'^([A-G](?:#|b)?)(.*)$', s)
        if not m:
            return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC:
        return None
    qual = normalize_quality(qual)
    return {'root': root, 'root_pc': NOTE_TO_PC[root], 'quality': qual, 'tones': chord_tones(NOTE_TO_PC[root], qual)}

In [ ]:
def find_jams_dir(data_root):
    candidates = [
        data_root / 'JamsFiles', data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c
    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None

def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir
    print('Could not find .jams files automatically.')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError(
        'Could not find any .jams files. Add the correct GuitarSet/FullGuitarSetData path to DATA_ROOT_CANDIDATES.'
    )

DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')

In [ ]:
def get_annotation_data(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []

def parse_string_from_data_source(data_source):
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None

def parse_jams_file(path):
    with open(path, 'r') as f:
        jam = json.load(f)
    notes, chords, beats = [], [], []
    tempo, key = None, None
    for ann in jam.get('annotations', []):
        ns = ann.get('namespace')
        rows = get_annotation_data(ann)
        data_source = ann.get('annotation_metadata', {}).get('data_source', '')
        if ns == 'note_midi':
            inferred_string = parse_string_from_data_source(data_source)
            for r in rows:
                v = r.get('value')
                if isinstance(v, dict):
                    midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                    string = v.get('string', inferred_string)
                    fret = v.get('fret')
                else:
                    midi = v
                    string = inferred_string
                    fret = None
                if midi is None:
                    continue
                midi_int = int(round(float(midi)))
                if string is not None and fret is None:
                    fret = midi_int - OPEN_STRING_MIDI[int(string)]
                notes.append({
                    'start': float(r.get('time', 0.0)),
                    'duration': float(r.get('duration', 0.0) or 0.0),
                    'midi': midi_int,
                    'pitch_class': midi_int % 12,
                    'true_string': None if string is None else int(string),
                    'true_fret': None if fret is None else int(round(float(fret))),
                    'source': data_source,
                })
        elif ns in ['chord', 'chord_harte']:
            for r in rows:
                start = float(r.get('time', 0.0))
                duration = float(r.get('duration', 0.0) or 0.0)
                chord_label = r.get('value')
                chords.append({
                    'start': start, 'duration': duration, 'end': start + duration,
                    'chord': chord_label, 'parsed': parse_chord_symbol(chord_label),
                })
        elif ns in ['beat', 'beat_position']:
            for r in rows:
                beats.append(float(r.get('time', 0.0)))
        elif ns == 'key_mode':
            if rows:
                key = rows[0].get('value')
        elif ns == 'tempo':
            if rows:
                tempo = rows[0].get('value')
    notes = sorted(notes, key=lambda x: (x['start'], x['midi']))
    chords = sorted(chords, key=lambda x: x['start'])
    return {'recording': path.stem, 'path': str(path), 'notes': notes, 'chords': chords, 'beats': beats, 'tempo': tempo, 'key': key}

records = [parse_jams_file(p) for p in JAMS_FILES]
print('Parsed records:', len(records))

## Key/context enrichment, onset grouping, playability/context costs

(identical to v2)

In [ ]:
def infer_key_from_filename(recording_name):
    parts = recording_name.split('_')
    if len(parts) >= 2:
        middle = parts[1]
        key_guess = middle.split('-')[-1]
        if key_guess in NOTE_TO_PC:
            return f'{key_guess} major'
    return None

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(':', ' ').strip()
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def chord_end_time(c):
    start = float(c.get('start', 0.0))
    if c.get('end') is not None:
        return float(c['end'])
    return start + float(c.get('duration', 0.0) or 0.0)

def chord_at_time(chords, t):
    for c in chords:
        start = float(c.get('start', 0.0))
        end = chord_end_time(c)
        if start <= t < end:
            return c
    return None

def enrich_notes_with_context(record):
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    key_info = get_key_info(key_label)
    out = []
    for n in record['notes']:
        c = chord_at_time(record['chords'], n['start'])
        row = dict(n)
        row['key_label'] = key_label
        row['in_key'] = None if key_info is None else (n['pitch_class'] in set(key_info['scale_pcs']))
        row['chord_label'] = None if c is None else c['chord']
        parsed_chord = None if c is None else c.get('parsed') or parse_chord_symbol(c.get('chord'))
        row['in_chord'] = None if parsed_chord is None else (n['pitch_class'] in set(parsed_chord['tones']))
        out.append(row)
    return out

def record_key_info(record):
    """Key dict (root_pc, mode, ...) for box_anchors_for_key, or None if unknown."""
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    return get_key_info(key_label)

In [ ]:
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position['fret']
    if fret == 0:
        return 0.0
    distance = abs(fret - hand_center)
    if distance <= 2:
        return 0.0
    if distance <= COMFORTABLE_SPAN:
        return 0.5 * (distance - 2)
    if distance <= MAX_REACHABLE_SPAN:
        return 2.0 + (distance - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (distance - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p['string'] for p in group_positions]
    frets = [p['fret'] for p in group_positions]
    fretted = [f for f in frets if f > 0]
    cost = 0.0
    if len(strings) != len(set(strings)):
        return float('inf')
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        string_span = max(strings) - min(strings)
        if string_span > 4 and len(strings) <= 3:
            cost += 1.5 * (string_span - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get('in_chord') is False:
            cost += 0.15
        if n.get('in_key') is False:
            cost += 0.10
    return cost

def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets = [p['fret'] for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center'] = estimate_hand_position_from_frets(frets)
    candidate['avg_string'] = float(np.mean(strings)) if strings else 0.0
    candidate['is_single'] = len(positions) == 1
    candidate['single_fret'] = positions[0]['fret'] if len(positions) == 1 else np.nan
    candidate['single_string'] = positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

## Old-theory score + `combined_all_tuned` (v1)

(identical to v2 / current `backend/fretboard.py`)

In [ ]:
OLD_THEORY_WEIGHTS = {
    'key_alignment': 1.0, 'chord_tone': 2.0, 'open_string_bonus': 1.0,
    'low_position_bonus': 0.5, 'middle_neck_bonus': 0.3,
    'position_continuity': 0.5, 'continuity_cap': 5.0,
}

def old_position_score(midi, position, note_row=None, previous_position=None, weights=None):
    if weights is None:
        weights = OLD_THEORY_WEIGHTS
    fret = position['fret']
    score = 0.0
    if note_row is not None and note_row.get('in_key') is True:
        score += weights['key_alignment']
    if note_row is not None and note_row.get('in_chord') is True:
        score += weights['chord_tone']
    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']
    if previous_position is not None:
        prev_fret = previous_position['fret']
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)
    return float(score)

def old_theory_group_cost(group_notes, group_positions):
    if not group_notes or not group_positions:
        return 0.0
    scores = []
    for n, p in zip(group_notes, group_positions):
        scores.append(old_position_score(n['midi'], p, note_row=n, previous_position=None))
    return -0.35 * float(np.mean(scores))

In [ ]:
def build_position_prior(records, alpha=0.50):
    counts = {}
    for rec in records:
        for n in rec.get('notes', []):
            midi, s, f = n.get('midi'), n.get('true_string'), n.get('true_fret')
            if midi is None or s is None or f is None:
                continue
            try:
                midi, s, f = int(midi), int(s), int(f)
            except Exception:
                continue
            if not (0 <= s < len(OPEN_STRING_MIDI) and 0 <= f <= MAX_FRET):
                continue
            if OPEN_STRING_MIDI[s] + f != midi:
                continue
            counts[(midi, s, f)] = counts.get((midi, s, f), 0) + 1

    prior_costs, prior_probs = {}, {}
    for midi in range(min(MIDI_TO_POSITIONS.keys()), max(MIDI_TO_POSITIONS.keys()) + 1):
        positions = get_possible_positions(midi)
        if not positions:
            continue
        total = sum(counts.get((midi, p['string'], p['fret']), 0) for p in positions)
        denom = total + alpha * len(positions)
        raw_costs = []
        for p in positions:
            prob = (counts.get((midi, p['string'], p['fret']), 0) + alpha) / denom
            cost = -math.log(prob)
            raw_costs.append(cost)
            prior_probs[(midi, p['string'], p['fret'])] = prob
        min_cost = min(raw_costs)
        for p, cost in zip(positions, raw_costs):
            prior_costs[(midi, p['string'], p['fret'])] = cost - min_cost
    return prior_costs, prior_probs

POSITION_PRIOR_COSTS, POSITION_PRIOR_PROBS = build_position_prior(records)

def position_prior_cost(midi, position):
    key = (int(midi), int(position['string']), int(position['fret']))
    return float(POSITION_PRIOR_COSTS.get(key, 0.75))

In [ ]:
DEFAULT_TUNED_WEIGHTS = {
    'playability': 0.70, 'context': 0.35, 'old_theory': 0.45, 'position_prior': 1.15,
    'hand_shift': 1.05, 'string_shift': 0.22, 'single_fret_shift': 0.65,
    'single_string_shift': 0.30, 'large_jump_extra': 4.50, 'open_after_high_extra': 2.25,
    'group_span_extra': 0.15,
}

def candidate_groups_combined_all_tuned(group_notes, weights=None, max_candidates=MAX_GROUP_CANDIDATES):
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        play_cost = group_playability_cost(combo)
        if not math.isfinite(play_cost):
            continue
        ctx_cost = context_cost(group_notes, combo)
        old_cost = old_theory_group_cost(group_notes, combo)
        prior_cost = float(np.mean([position_prior_cost(n['midi'], p) for n, p in zip(group_notes, combo)]))
        frets = [p['fret'] for p in combo]
        span_extra = group_span(frets)
        base_cost = (
            weights['playability'] * play_cost
            + weights['context'] * ctx_cost
            + weights['old_theory'] * old_cost
            + weights['position_prior'] * prior_cost
            + weights['group_span_extra'] * span_extra
        )
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({
                'positions': combo, 'base_cost': float(base_cost),
                'playability_cost': float(play_cost), 'context_cost': float(ctx_cost),
                'old_theory_cost': float(old_cost), 'position_prior_cost': float(prior_cost),
            }))
    if not candidates:
        # extreme fallback: any physically valid combo
        for combo in product(*position_lists):
            combo = list(combo)
            if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
                continue
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': 50.0, 'position_prior_cost': 0.75}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

def tuned_transition_cost_matrix(prev_cands, curr_cands, weights=None):
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)
    mat = weights['hand_shift'] * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += weights['string_shift'] * np.abs(prev_str[:, None] - curr_str[None, :])
    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]
    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]
        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = weights['single_fret_shift'] * fret_diff
        extra += weights['single_string_shift'] * string_diff
        extra += np.where(fret_diff > LARGE_JUMP_THRESHOLD, weights['large_jump_extra'] + fret_diff - LARGE_JUMP_THRESHOLD, 0.0)
        extra += np.where((cf == 0) & (pf > 7), weights['open_after_high_extra'], 0.0)
        mat += np.where(single_mask, extra, 0.0)
    return mat

def assign_combined_all_tuned_with_weights(notes, weights=None, method_name='combined_all_tuned'):
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all_tuned(g, weights=weights) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]
    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = tuned_transition_cost_matrix(prev_cands, curr_cands, weights=weights)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': method_name})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))

def assign_combined_all_tuned(notes):
    return assign_combined_all_tuned_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS, method_name='combined_all_tuned')

## Second-order position prior (v2) -- `P(string, fret | midi, prev_string_bucket)`

In [ ]:
def _string_bucket(avg_string):
    """Bucket an average string index (0=low E ... 5=high E) into 3 coarse zones."""
    if avg_string is None or (isinstance(avg_string, float) and math.isnan(avg_string)):
        return -1
    if avg_string <= 1.5:
        return 0
    if avg_string <= 3.5:
        return 1
    return 2

def build_position_prior_2nd(records, alpha=0.50):
    counts = {}
    for rec in records:
        notes = enrich_notes_with_context(rec)
        notes = [n for n in notes if n.get('true_string') is not None and n.get('true_fret') is not None and 0 <= n['true_fret'] <= MAX_FRET]
        if not notes:
            continue
        groups = group_notes_by_onset(notes)
        prev_bucket = -1
        for g in groups:
            for n in g:
                midi, s, f = n.get('midi'), n.get('true_string'), n.get('true_fret')
                if midi is None or s is None or f is None:
                    continue
                try:
                    midi, s, f = int(midi), int(s), int(f)
                except Exception:
                    continue
                if not (0 <= s < len(OPEN_STRING_MIDI) and 0 <= f <= MAX_FRET):
                    continue
                if OPEN_STRING_MIDI[s] + f != midi:
                    continue
                counts[(midi, prev_bucket, s, f)] = counts.get((midi, prev_bucket, s, f), 0) + 1
            true_strings = [n['true_string'] for n in g if n.get('true_string') is not None]
            if true_strings:
                prev_bucket = _string_bucket(float(np.mean(true_strings)))

    by_context = defaultdict(dict)
    for (midi, bucket, s, f), c in counts.items():
        by_context[(midi, bucket)][(s, f)] = c

    prior_costs_2nd = {}
    for (midi, bucket), pos_counts in by_context.items():
        positions = get_possible_positions(midi)
        if not positions:
            continue
        total = sum(pos_counts.get((p['string'], p['fret']), 0) for p in positions)
        denom = total + alpha * len(positions)
        raw_costs = []
        for p in positions:
            prob = (pos_counts.get((p['string'], p['fret']), 0) + alpha) / denom
            raw_costs.append(-math.log(prob))
        min_cost = min(raw_costs)
        for p, cost in zip(positions, raw_costs):
            prior_costs_2nd[(midi, bucket, p['string'], p['fret'])] = cost - min_cost
    return prior_costs_2nd

POSITION_PRIOR_COSTS_2ND = build_position_prior_2nd(records)

def position_prior_cost_2nd(midi, position, prev_bucket):
    key = (int(midi), prev_bucket, int(position['string']), int(position['fret']))
    if key in POSITION_PRIOR_COSTS_2ND:
        return float(POSITION_PRIOR_COSTS_2ND[key])
    return position_prior_cost(midi, position)

In [ ]:
DEFAULT_TUNED_WEIGHTS_V2 = {
    **DEFAULT_TUNED_WEIGHTS,
    'position_prior_2nd': 0.40,
    'prior_2nd_mix': 0.60,
}

def candidate_groups_combined_all_tuned_v2(group_notes, weights=None, max_candidates=MAX_GROUP_CANDIDATES):
    """Same candidates as combined_all_tuned; the v2 logic lives in the transition matrix."""
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS_V2
    return candidate_groups_combined_all_tuned(group_notes, weights=weights, max_candidates=max_candidates)

def tuned_transition_cost_matrix_v2(prev_cands, curr_cands, curr_group_notes, weights=None):
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS_V2
    mat = tuned_transition_cost_matrix(prev_cands, curr_cands, weights=weights)
    mix = weights.get('prior_2nd_mix', 0.0)
    w2 = weights.get('position_prior_2nd', 0.0)
    if w2 == 0.0:
        return mat
    prev_buckets = [_string_bucket(c['avg_string']) for c in prev_cands]
    for j, cand in enumerate(curr_cands):
        first_order = cand.get('position_prior_cost', 0.0)
        for i, prev_bucket in enumerate(prev_buckets):
            second_order = float(np.mean([
                position_prior_cost_2nd(n['midi'], p, prev_bucket)
                for n, p in zip(curr_group_notes, cand['positions'])
            ]))
            blended = mix * second_order + (1.0 - mix) * first_order
            mat[i, j] += w2 * (blended - first_order)
    return mat

def assign_combined_all_tuned_v2_with_weights(notes, weights=None, method_name='combined_all_tuned_v2'):
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS_V2
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all_tuned_v2(g, weights=weights) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]
    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = tuned_transition_cost_matrix_v2(prev_cands, curr_cands, groups[i], weights=weights)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': method_name})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))

def assign_combined_all_tuned_v2(notes):
    return assign_combined_all_tuned_v2_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS_V2, method_name='combined_all_tuned_v2')

print('v1 + v2 (second-order prior) methods defined.')

## CAGED-box hand-position windows (Ani's approach)

A "hand position" is a `BOX_WINDOW`-fret window `[anchor, anchor+BOX_WINDOW]`.
For the detected key we mark which windows line up with real pentatonic
CAGED boxes, and which one is the "home" box (root on the low E string).

In [ ]:
PENTATONIC = {'major': [0, 2, 4, 7, 9], 'minor': [0, 3, 5, 7, 10]}
LOW_E_PC = OPEN_STRING_MIDI[0] % 12  # 4 (E)

def box_anchors_for_key(key, max_fret=MAX_FRET, window=BOX_WINDOW):
    """List of candidate hand-position windows with a key-fit cost each."""
    anchors_range = range(0, max_fret - window + 1)
    if key is None:
        return [{'anchor': a, 'home': False, 'key_cost': BOX_LOWNECK_COST * a} for a in anchors_range]

    r = key['root_pc']
    penta = PENTATONIC.get(key['mode'], PENTATONIC['minor'])

    box_anchor_frets = set()
    for deg in penta:
        f = (deg + (r - LOW_E_PC)) % 12
        while f <= max_fret - 1:
            box_anchor_frets.add(f)
            f += 12

    home_anchors = set()
    h = (r - LOW_E_PC) % 12
    while h <= max_fret - 1:
        home_anchors.add(h)
        h += 12

    out = []
    for a in anchors_range:
        d_box = min((abs(a - b) for b in box_anchor_frets), default=0)
        is_home = a in home_anchors
        key_cost = (BOX_OFFBOX_COST * d_box + (0.0 if is_home else BOX_NONHOME_COST) + BOX_LOWNECK_COST * a)
        out.append({'anchor': a, 'home': is_home, 'key_cost': key_cost})
    return out

def position_window_cost(p, anchor, window=BOX_WINDOW):
    """Cost of placing one note at position p given the active hand window."""
    f = p['fret']
    if f == 0:
        return 0.0 if anchor <= 2 else OPEN_OUT_OF_BOX_COST
    if anchor <= f <= anchor + window:
        return BOX_CENTER_COST * abs(f - (anchor + window / 2.0))
    dist = (anchor - f) if f < anchor else (f - (anchor + window))
    return BOX_OUTSIDE_COST * dist

def candidate_window_cost(cand, anchor):
    return sum(position_window_cost(p, anchor) for p in cand['positions'])

### `caged_box`: candidates scored only by playability/context (no learned prior),
Viterbi state = hand-position window. (Reproduced from Ani's notebook, with
`MAX_CHORD_SPAN`-style hard wall folded into `group_playability_cost` above
via `MAX_REACHABLE_SPAN`.)

In [ ]:
CAGED_BOX_ONLY_WEIGHTS = {'playability': 0.80, 'context': 0.30, 'box_window': 1.00, 'hand_move': 0.70}

def candidate_groups_caged_only(group_notes, weights=None, max_candidates=MAX_GROUP_CANDIDATES):
    if weights is None:
        weights = CAGED_BOX_ONLY_WEIGHTS
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        play = group_playability_cost(combo)
        if not math.isfinite(play):
            continue
        base = weights['playability'] * play + weights['context'] * context_cost(group_notes, combo)
        candidates.append(enrich_candidate({'positions': combo, 'base_cost': float(base)}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
                continue
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': 50.0}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

def assign_caged_box(notes, key=None, weights=None):
    if weights is None:
        weights = CAGED_BOX_ONLY_WEIGHTS
    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    all_cands = [candidate_groups_caged_only(g, weights=weights) for g in groups]
    if any(len(c) == 0 for c in all_cands):
        raise ValueError('An onset group had no playable candidate positions.')

    anchors = box_anchors_for_key(key)
    A = len(anchors)
    anchor_fret = np.array([a['anchor'] for a in anchors], dtype=float)
    n = len(groups)

    emit_cost = np.empty((n, A), dtype=float)
    emit_cand = [[0] * A for _ in range(n)]
    for i, cands in enumerate(all_cands):
        for j, anc in enumerate(anchors):
            a = anc['anchor']
            best, best_ci = None, 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + weights['box_window'] * candidate_window_cost(c, a)
                if best is None or tot < best:
                    best, best_ci = tot, ci
            emit_cost[i, j] = best + anc['key_cost']
            emit_cand[i][j] = best_ci

    delta = np.abs(anchor_fret[:, None] - anchor_fret[None, :])
    shift = np.maximum(delta - SHIFT_FREE, 0.0)
    big = np.maximum(delta - LARGE_JUMP_THRESHOLD, 0.0)
    trans = weights['hand_move'] * shift + 0.5 * big ** 2

    dp = np.empty((n, A))
    back = np.zeros((n, A), dtype=int)
    dp[0] = emit_cost[0]
    back[0] = -1
    for i in range(1, n):
        scores = dp[i - 1][:, None] + trans + emit_cost[i][None, :]
        back[i] = np.argmin(scores, axis=0)
        dp[i] = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1]))
    chosen = [j]
    for i in range(n - 1, 0, -1):
        j = int(back[i][j])
        chosen.append(j)
    chosen = list(reversed(chosen))

    pred = []
    for i, (g, cands) in enumerate(zip(groups, all_cands)):
        aj = chosen[i]
        c = cands[emit_cand[i][aj]]
        for note, p in zip(g, c['positions']):
            row = dict(note)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'caged_box', 'anchor': anchors[aj]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))

print('caged_box defined.')

## Combined method: CAGED hand-position state + first/second-order learned priors

`assign_combined_caged_v2`:
- **State space** = CAGED hand-position window/anchor (Ani's idea) -- this is
  what makes "stay in this box until the music really moves" the default,
  instead of re-evaluating only the previous note.
- **Candidate scoring within a window** = `combined_all_tuned`'s full cost
  (playability + context + old-theory + *first-order* learned position
  prior + group-span) **plus** the CAGED window-fit cost for that anchor.
- **Transition between windows** = CAGED's hand-move cost (free up to
  `SHIFT_FREE` frets, then growing) **plus** a *second-order* prior
  correction: given the previous window's chosen candidate (and its average
  string register -> low/mid/high bucket), how typical is the current
  window's chosen candidate for its MIDI notes? This mirrors v2's
  transition-time correction, but the "previous context" is now anchored to
  the previous *hand position* rather than just the previous note.

In [ ]:
DEFAULT_COMBINED_CAGED_V2_WEIGHTS = {
    **DEFAULT_TUNED_WEIGHTS_V2,
    'box_window': 1.00,
    'hand_move': 0.70,
}

def assign_combined_caged_v2(notes, key=None, weights=None):
    if weights is None:
        weights = DEFAULT_COMBINED_CAGED_V2_WEIGHTS

    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    all_cands = [candidate_groups_combined_all_tuned_v2(g, weights=weights) for g in groups]
    if any(len(c) == 0 for c in all_cands):
        raise ValueError('An onset group had no playable candidate positions.')

    anchors = box_anchors_for_key(key)
    A = len(anchors)
    anchor_fret = np.array([a['anchor'] for a in anchors], dtype=float)
    n = len(groups)

    # Emission: best in-window candidate per (group, anchor), using the full
    # v1/v2 base_cost (which already includes the first-order position prior)
    # plus the CAGED window-fit cost.
    emit_cost = np.empty((n, A), dtype=float)
    emit_cand = [[0] * A for _ in range(n)]
    for i, cands in enumerate(all_cands):
        for j, anc in enumerate(anchors):
            a = anc['anchor']
            best, best_ci = None, 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + weights['box_window'] * candidate_window_cost(c, a)
                if best is None or tot < best:
                    best, best_ci = tot, ci
            emit_cost[i, j] = best + anc['key_cost']
            emit_cand[i][j] = best_ci

    # Base transition: CAGED hand-move cost between anchors.
    delta = np.abs(anchor_fret[:, None] - anchor_fret[None, :])
    shift = np.maximum(delta - SHIFT_FREE, 0.0)
    big = np.maximum(delta - LARGE_JUMP_THRESHOLD, 0.0)
    base_trans = weights['hand_move'] * shift + 0.5 * big ** 2

    mix = weights.get('prior_2nd_mix', 0.0)
    w2 = weights.get('position_prior_2nd', 0.0)

    dp = np.empty((n, A))
    back = np.zeros((n, A), dtype=int)
    dp[0] = emit_cost[0]
    back[0] = -1

    for i in range(1, n):
        trans = base_trans.copy()
        if w2 != 0.0:
            cands_i = all_cands[i]
            cands_prev = all_cands[i - 1]
            for jj in range(A):
                cand_j = cands_i[emit_cand[i][jj]]
                first_order = cand_j.get('position_prior_cost', 0.0)
                for ii in range(A):
                    cand_i = cands_prev[emit_cand[i - 1][ii]]
                    prev_bucket = _string_bucket(cand_i['avg_string'])
                    second_order = float(np.mean([
                        position_prior_cost_2nd(note['midi'], p, prev_bucket)
                        for note, p in zip(groups[i], cand_j['positions'])
                    ]))
                    blended = mix * second_order + (1.0 - mix) * first_order
                    trans[ii, jj] += w2 * (blended - first_order)

        scores = dp[i - 1][:, None] + trans + emit_cost[i][None, :]
        back[i] = np.argmin(scores, axis=0)
        dp[i] = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1]))
    chosen = [j]
    for i in range(n - 1, 0, -1):
        j = int(back[i][j])
        chosen.append(j)
    chosen = list(reversed(chosen))

    pred = []
    for i, (g, cands) in enumerate(zip(groups, all_cands)):
        aj = chosen[i]
        c = cands[emit_cand[i][aj]]
        for note, p in zip(g, c['positions']):
            row = dict(note)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'combined_caged_v2', 'anchor': anchors[aj]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))

print('combined_caged_v2 defined.')

## Evaluation helpers

(identical to v2)

In [ ]:
def add_prediction_diagnostics(df):
    df = df.copy()
    df['pred_midi'] = [OPEN_STRING_MIDI[int(s)] + int(f) if pd.notna(s) and pd.notna(f) else np.nan for s, f in zip(df['pred_string'], df['pred_fret'])]
    df['correct_pitch_from_tab'] = df['pred_midi'] == df['midi']
    df['valid_position'] = df.apply(lambda r: pd.notna(r['pred_string']) and pd.notna(r['pred_fret']) and 0 <= int(r['pred_string']) <= 5 and 0 <= int(r['pred_fret']) <= MAX_FRET, axis=1)
    df['exact_position_correct'] = (df['pred_string'] == df['true_string']) & (df['pred_fret'] == df['true_fret'])
    df['string_correct'] = df['pred_string'] == df['true_string']
    df['fret_correct'] = df['pred_fret'] == df['true_fret']
    df['fret_error'] = (df['pred_fret'] - df['true_fret']).abs()
    df['string_error'] = (df['pred_string'] - df['true_string']).abs()
    return df

def duplicate_string_violation_rate(df, onset_tolerance=ONSET_TOLERANCE_SECONDS):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'), tolerance=onset_tolerance)
    violations, total_chord_groups = 0, 0
    for g in groups:
        if len(g) <= 1:
            continue
        total_chord_groups += 1
        strings = [x.get('pred_string') for x in g if pd.notna(x.get('pred_string'))]
        if len(strings) != len(set(strings)):
            violations += 1
    return violations / total_chord_groups if total_chord_groups else 0.0

def average_group_span(df):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    spans = []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        if frets:
            spans.append(group_span(frets))
    return float(np.mean(spans)) if spans else np.nan

def movement_metrics(df):
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    centers, avg_strings = [], []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        strings = [int(x['pred_string']) for x in g if pd.notna(x.get('pred_string'))]
        if frets and strings:
            centers.append(estimate_hand_position_from_frets(frets))
            avg_strings.append(float(np.mean(strings)))
    if len(centers) <= 1:
        return {'avg_fret_jump': 0.0, 'avg_string_jump': 0.0, 'large_jump_rate': 0.0, 'large_jump_count': 0}
    fret_jumps = np.abs(np.diff(centers))
    string_jumps = np.abs(np.diff(avg_strings))
    large = fret_jumps > LARGE_JUMP_THRESHOLD
    return {'avg_fret_jump': float(np.mean(fret_jumps)), 'avg_string_jump': float(np.mean(string_jumps)), 'large_jump_rate': float(np.mean(large)), 'large_jump_count': int(np.sum(large))}

def evaluate_predictions(pred_rows):
    df = pd.DataFrame(pred_rows)
    if df.empty:
        return {}, df
    df = add_prediction_diagnostics(df)
    mv = movement_metrics(df)
    metrics = {
        'n_notes': len(df),
        'exact_position_acc': float(df['exact_position_correct'].mean()),
        'string_acc': float(df['string_correct'].mean()),
        'fret_acc': float(df['fret_correct'].mean()),
        'avg_fret_error': float(df['fret_error'].mean()),
        'avg_string_error': float(df['string_error'].mean()),
        'correct_pitch_from_tab_rate': float(df['correct_pitch_from_tab'].mean()),
        'valid_position_rate': float(df['valid_position'].mean()),
        'duplicate_string_violation_rate': float(duplicate_string_violation_rate(df)),
        'avg_group_span': float(average_group_span(df)),
        **mv,
    }
    return metrics, df

## Leave-one-out cross-validation: v1 vs v2 vs caged_box vs combined_caged_v2

For each recording, rebuild both position priors from the other N-1
recordings, then evaluate all four methods on the held-out recording.
Averaging across folds gives a leakage-free accuracy estimate even with a
small number of recordings (this local sample has 9).

In [ ]:
print(f'Running leave-one-out CV over {len(records)} recordings...')
loocv_rows = []
for i, test_rec in enumerate(records):
    train_recs = [r for j, r in enumerate(records) if j != i]

    fold_prior_1st, _ = build_position_prior(train_recs)
    fold_prior_2nd = build_position_prior_2nd(train_recs)

    saved_1st, saved_2nd = POSITION_PRIOR_COSTS, POSITION_PRIOR_COSTS_2ND
    POSITION_PRIOR_COSTS, POSITION_PRIOR_COSTS_2ND = fold_prior_1st, fold_prior_2nd

    try:
        notes = enrich_notes_with_context(test_rec)
        notes = [n for n in notes if n.get('true_string') is not None and n.get('true_fret') is not None and 0 <= n['true_fret'] <= MAX_FRET]
        if not notes:
            continue
        key = record_key_info(test_rec)

        pred_v1 = assign_combined_all_tuned_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS, method_name='combined_all_tuned')
        metrics_v1, _ = evaluate_predictions(pred_v1)

        pred_v2 = assign_combined_all_tuned_v2_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS_V2, method_name='combined_all_tuned_v2')
        metrics_v2, _ = evaluate_predictions(pred_v2)

        pred_caged = assign_caged_box(notes, key=key, weights=CAGED_BOX_ONLY_WEIGHTS)
        metrics_caged, _ = evaluate_predictions(pred_caged)

        pred_combined = assign_combined_caged_v2(notes, key=key, weights=DEFAULT_COMBINED_CAGED_V2_WEIGHTS)
        metrics_combined, _ = evaluate_predictions(pred_combined)
    finally:
        POSITION_PRIOR_COSTS, POSITION_PRIOR_COSTS_2ND = saved_1st, saved_2nd

    for method_name, m in [
        ('combined_all_tuned', metrics_v1),
        ('combined_all_tuned_v2', metrics_v2),
        ('caged_box', metrics_caged),
        ('combined_caged_v2', metrics_combined),
    ]:
        loocv_rows.append({
            'recording': test_rec['recording'],
            'method': method_name,
            'exact_position_acc': m['exact_position_acc'],
            'string_acc': m['string_acc'],
            'fret_acc': m['fret_acc'],
            'avg_fret_error': m['avg_fret_error'],
            'avg_string_error': m['avg_string_error'],
            'avg_fret_jump': m['avg_fret_jump'],
            'large_jump_rate': m['large_jump_rate'],
            'duplicate_string_violation_rate': m['duplicate_string_violation_rate'],
            'avg_group_span': m['avg_group_span'],
        })
    print(f"  fold {i+1}/{len(records)} {test_rec['recording']}: "
          f"v1={metrics_v1['exact_position_acc']:.4f}  v2={metrics_v2['exact_position_acc']:.4f}  "
          f"caged={metrics_caged['exact_position_acc']:.4f}  combined={metrics_combined['exact_position_acc']:.4f}")

loocv_df = pd.DataFrame(loocv_rows)
loocv_path = OUTPUT_DIR / 'fretboard_loocv_results_combined_caged_v2.csv'
loocv_df.to_csv(loocv_path, index=False)

loocv_summary = loocv_df.groupby('method')[[
    'exact_position_acc', 'string_acc', 'fret_acc', 'avg_fret_error', 'avg_string_error',
    'avg_fret_jump', 'large_jump_rate', 'duplicate_string_violation_rate', 'avg_group_span',
]].mean().sort_values('exact_position_acc', ascending=False)

print('\nLOOCV averages (true held-out, no leakage):')
print(loocv_summary)
print('\nSaved per-fold LOOCV results to:', loocv_path.resolve())

# Rebuild full-data priors for any later cells that reference module-level priors.
POSITION_PRIOR_COSTS, POSITION_PRIOR_PROBS = build_position_prior(records)
POSITION_PRIOR_COSTS_2ND = build_position_prior_2nd(records)

## Pipeline-level accuracy estimate

`exact_position_acc` only measures fretboard assignment given perfect note
detection. Multiply by the Basic Pitch note-detection F1 (0.736, from
`jupyter_notebooks/eval_pipeline.ipynb` on this same 9-recording sample) for
a rough end-to-end estimate.

In [ ]:
NOTE_DETECTION_F1 = 0.736

pipeline_df = loocv_summary[['exact_position_acc']].copy()
pipeline_df['note_detection_f1'] = NOTE_DETECTION_F1
pipeline_df['pipeline_acc_estimate'] = pipeline_df['exact_position_acc'] * NOTE_DETECTION_F1
print(pipeline_df)

pipeline_path = OUTPUT_DIR / 'pipeline_accuracy_estimate_combined_caged_v2.csv'
pipeline_df.to_csv(pipeline_path)
print('\nSaved pipeline accuracy estimate to:', pipeline_path.resolve())